# Wordprint on Colab

用 Google 的 GPU 跑 Qwen 轉錄並分說話者，不必在自己電腦安裝模型，也不需要 Hugging Face 帳號或 token。音檔會上傳到這個 Colab 執行環境（Google 的機器）。

1. 「執行階段 → 全部執行」，在第 3 格選音檔上傳。不用重新啟動。
2. 最後會下載 `normalized-run-*.json`；交給 Wordprint 繼續（驗證、閱讀頁、摘要、郵件）。

預設用 T4 GPU；沒分配到 GPU 時照樣用 CPU 跑，並自動改用較快的 0.6B 模型。說話者由 NVIDIA Nemotron-3-Diarization 標記（最多 8 人）。

不在 Colab（本機 Jupyter，未實測）：第 1 格填 `AUDIO_FILES`，結果寫在目前資料夾。

In [ ]:
MODEL_SIZE = "1.7B"  # 或 "0.6B"（較快、堪用）；沒有 GPU 時自動改用 0.6B
MERGE = False  # 一次上傳多個檔案時，True＝依上傳順序合併成一段；False＝每個檔案各自一份結果
MAX_NEW_TOKENS = 2048  # 每段（≤3 分鐘）輸出上限；碰到上限的結果會標成 truncated
AUDIO_FILES = []  # 只在本機 Jupyter 用：音檔路徑（Colab 改用上傳）

In [ ]:
# 一次解析全部版本：qwen-asr 鎖定 transformers 4.57.6，Nemotron 需要 NeMo 的這個版本。
# pip 可能印出 google-adk、diffusers 等 Colab 內建套件的相依衝突 ERROR；本筆記本用不到它們，可以忽略。
!pip install -q qwen-asr transformers==4.57.6 Cython packaging "nemo-toolkit[asr] @ https://github.com/NVIDIA/NeMo/archive/16b0ff2584c6355c53cbd6c28f0f431a76b24956.zip"

In [ ]:
import shlex, sys
from pathlib import Path
try:
    from google.colab import files
    names = list(files.upload())  # 檔案存到目前目錄
except ImportError:  # 本機 Jupyter
    files, names = None, AUDIO_FILES
groups = [names] if MERGE else [[name] for name in names]
print("將轉錄：", groups)

In [ ]:
%%writefile worker_runtime.py
"""Small, offline checkpoint contract shared by Wordprint workers."""
import hashlib
import json
import math
from fractions import Fraction
from pathlib import Path
import shutil
import wave


def bound_alignment_terminal_bin(words, *, sample_count, sample_rate, quantum_seconds):
    """Intersect only an exact terminal model bin with PCM; never repair invalid ordering.

    Words use local seconds (start/end). Unknown quantum leaves them unchanged.
    Equality is exact: there is no millisecond tolerance or nearest-bin rounding.
    """
    result = [dict(word) for word in words]
    if (type(sample_count) is not int or sample_count <= 0
            or type(sample_rate) is not int or sample_rate <= 0
            or type(quantum_seconds) not in (int, float)
            or not math.isfinite(quantum_seconds) or quantum_seconds <= 0):
        return result, []
    # PCM and decimal model milliseconds define exact rational clocks. Float
    # ceil(.56 / .08) is 8, incorrectly admitting the following .64-second bin.
    quantum = Fraction(str(quantum_seconds))
    duration_exact = Fraction(sample_count, sample_rate)
    duration = float(duration_exact)
    # Convert only once, matching the provider's decimal millisecond timestamps
    # (e.g. 29.76), not repeated binary float multiplication (e.g. 29.040000000000003).
    terminal_end = float(math.ceil(duration_exact / quantum) * quantum)
    previous_start = previous_end = 0
    for word in words:
        start, end = word.get('start'), word.get('end')
        if (type(start) not in (int, float) or type(end) not in (int, float)
                or not math.isfinite(start) or not math.isfinite(end)
                or not 0 <= start <= end or start > duration
                or start < previous_start or end < previous_end):
            return result, []
        previous_start, previous_end = start, end
    adjustments = []
    for index, word in enumerate(result):
        if word['start'] < duration < word['end'] == terminal_end:
            adjustments.append({'word_index': index, 'start': word['start'],
                                'original_end': word['end'], 'bounded_end': duration})
            word['end'] = duration
    # Intersecting a later word must not move its end before an earlier,
    # non-terminal prediction. Roll back the whole adaptation in that case;
    # canonical validation must see the original invalid boundary, not hide it
    # inside its existing rounding tolerance.
    if any(right['end'] < left['end'] for left, right in zip(result, result[1:])):
        return [dict(word) for word in words], []
    return result, adjustments


def save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)


def digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def state(directory, value):
    value.setdefault('warnings', [])
    value.setdefault('status', 'running')
    value.setdefault('retry_stage', None)
    value['available_outputs'] = sorted(p.name for p in Path(directory).iterdir()
                                        if p.is_file() and p.suffix in ('.json', '.txt', '.wav', '.log')
                                        and p.name != 'state.json')
    save(Path(directory) / 'state.json', value)


def raw_text(directory, texts):
    text = '\n'.join(text for text in texts if text)
    if text:
        path = Path(directory) / 'raw-transcript.txt'
        temporary = path.with_suffix('.txt.tmp')
        temporary.write_text(text + '\n', encoding='utf-8')
        temporary.replace(path)


def finish_error(directory, value, error):
    cancelled = isinstance(error, KeyboardInterrupt)
    stage = value.get('stage', 'start')
    value.update(stage='cancelled' if cancelled else 'failed',
                 status='cancelled' if cancelled else ('partial' if (Path(directory) / 'raw-transcript.txt').exists() else 'failed'),
                 retry_stage=stage, error='使用者已取消；已完成的輸出仍保留。' if cancelled else str(error))
    state(directory, value)


def check_location(old, new):
    if old:
        old, new = Path(old).resolve(), Path(new).resolve()
        if old == new or old in new.parents:
            raise ValueError('續跑必須使用舊 run 以外的新目錄；既有目錄未改動。')


def checkpoint(directory, config, sources, stage, files):
    path = Path(directory) / 'checkpoint.json'
    record = json.loads(path.read_text()) if path.exists() else {
        'version': 1, 'config': config, 'sources': [s['sha256'] for s in sources], 'stages': [], 'artifacts': {}}
    for name in files:
        record['artifacts'][name] = digest(Path(directory) / name)
    if stage not in record['stages']:
        record['stages'].append(stage)
    save(path, record)
    return record


def resume(old, new, sources, config):
    """Validate everything before copying; never alter the previous run or silently fall back to ASR."""
    old, new = Path(old).resolve(), Path(new).resolve()
    check_location(old, new)
    record = json.loads((old / 'checkpoint.json').read_text(encoding='utf-8'))
    if record.get('version') != 1 or record.get('config') != config:
        raise ValueError('續跑模型或設定不符，不能重用 checkpoint。')
    if record.get('sources') != [digest(source) for source in sources]:
        raise ValueError('續跑來源雜湊或順序不符。')
    artifacts = record.get('artifacts', {})
    if not artifacts or not {'canonical.wav', 'sources.json'} <= artifacts.keys():
        raise ValueError('checkpoint 缺少 canonical audio 或來源記錄。')
    for name, expected in artifacts.items():
        path = old / name
        if Path(name).name != name or path.is_symlink() or not path.is_file() or digest(path) != expected:
            raise ValueError(f'checkpoint 檔案完整性不符：{name}')
    records = json.loads((old / 'sources.json').read_text(encoding='utf-8'))
    if [s['sha256'] for s in records] != record['sources']:
        raise ValueError('checkpoint 來源記錄不符。')
    pcm = read_pcm(old / 'canonical.wav')
    if sum(s['sample_count'] for s in records) * 2 != len(pcm):
        raise ValueError('checkpoint PCM 樣本數不符。')
    # Read the request journal before publishing any resumable checkpoint. A new
    # run interrupted during the copy must never look like no request was sent.
    prior_state = json.loads((old / 'state.json').read_text(encoding='utf-8'))
    if config.get('backend') == 'Gemini Interactions API v1beta' and (
            not isinstance(prior_state.get('part_requests'), dict) or
            not isinstance(prior_state.get('google_resources'), list)):
        raise ValueError('Gemini 續跑缺少請求或資源紀錄；停止自動重送。')
    for name in artifacts:
        shutil.copyfile(old / name, new / name)
    # Paths may move; ordered source bytes, rather than a previous machine's paths, define identity.
    for item, source in zip(records, sources):
        item['path'] = str(Path(source).resolve())
    save(new / 'sources.json', records)
    record['artifacts']['sources.json'] = digest(new / 'sources.json')
    save(new / 'resume-state.json', prior_state)
    state(new, {**prior_state, 'stage': 'resume', 'status': 'running', 'resumed_from': str(old)})
    # Publish last. If interrupted before this point, the new run is not resumable;
    # afterwards its durable state already includes every prior request/resource.
    save(new / 'checkpoint.json', record)
    return records, pcm, record, prior_state


def read_pcm(path):
    with wave.open(str(path), 'rb') as audio:
        if (audio.getnchannels(), audio.getsampwidth(), audio.getframerate()) != (1, 2, 16000):
            raise ValueError('checkpoint PCM 格式不符。')
        frames = audio.getnframes()
        pcm = audio.readframes(frames)
    if frames <= 0 or len(pcm) != frames * 2:
        raise ValueError('checkpoint PCM 不完整。')
    return pcm


def write_pcm(path, pcm):
    with wave.open(str(path), 'wb') as audio:
        audio.setnchannels(1), audio.setsampwidth(2), audio.setframerate(16000)
        audio.writeframes(pcm)


In [ ]:
%%writefile transcript.py
#!/usr/bin/env python3
"""Validate normalized ASR/alignment/diarization results and render losslessly.

This is a postprocessor, not an inference engine. All times are global seconds.
Use --help for the CLI. Input and output JSON preserve original model text.
"""
import argparse
import hashlib
import html
import json
import math
import re
import shutil
import subprocess
import sys
import tempfile
import unicodedata
import urllib.parse
import wave
from pathlib import Path

EPS = 0.02  # Accommodate rounding to centiseconds, never invent missing audio.
EMPTY_TRANSCRIPT_NOTICE = '未辨識到可用文字。請先回聽原錄音確認語音是否清楚，再重新選擇來源或嘗試其他模型。'


class Invalid(ValueError):
    pass


def require(condition, message):
    if not condition:
        raise Invalid(message)


def number(value, where):
    require(type(value) in (int, float) and math.isfinite(value), f'{where}: finite number required')
    return value


def string(value, where, nonempty=True):
    require(isinstance(value, str), f'{where}: string required')
    require(not nonempty or bool(value.strip()), f'{where}: empty string')
    return value


def normalized(text):
    """Ignore only whitespace, Unicode punctuation and case, not symbols/digits."""
    return ''.join(c.casefold() for c in text if not c.isspace() and not unicodedata.category(c).startswith('P'))


def object_pairs(pairs):
    result = {}
    for key, value in pairs:
        require(key not in result, f'duplicate JSON key: {key}')
        result[key] = value
    return result


def interval(item, where, limit):
    require(isinstance(item, dict), f'{where}: object required')
    start, end = number(item.get('start'), f'{where}.start'), number(item.get('end'), f'{where}.end')
    require(0 <= start <= end <= limit + EPS, f'{where}: invalid time interval')
    return start, end


def validate(path, require_speakers=True):
    """require_speakers=False checks an alignment-only (partial) run: everything except diarization_complete."""
    path = Path(path).resolve()
    run = json.loads(path.read_text(encoding='utf-8-sig'), object_pairs_hook=object_pairs)
    require(isinstance(run, dict) and type(run.get('schema_version')) is int and run['schema_version'] == 1,
            'schema_version must be 1')
    model = run.get('model')
    require(isinstance(model, dict), 'model: object required')
    string(model.get('id'), 'model.id')
    named_sizes = {size for size in ('0.6B', '1.7B') if size.casefold() in model['id'].casefold()}
    # Non-Qwen engines (e.g. gemini-3.5-transcribe) have no size; a Qwen size in the id still requires it.
    require(model.get('size') in ('0.6B', '1.7B') or (model.get('size') is None and not named_sizes),
            'model.size must be 0.6B or 1.7B (null only for an id naming no Qwen size)')
    require(not named_sizes or named_sizes == {model['size']}, 'model.id and model.size disagree')
    require(not require_speakers or run.get('diarization_complete') is True, 'diarization_complete must be true; partial runs cannot render as complete')
    sources = run.get('sources')
    require(isinstance(sources, list) and sources, 'sources: nonempty array required')
    duration = 0.0
    for i, source in enumerate(sources):
        where = f'sources[{i}]'
        require(isinstance(source, dict), f'{where}: object required')
        source_path = Path(string(source.get('path'), f'{where}.path'))
        source_path = (path.parent / source_path).resolve() if not source_path.is_absolute() else source_path.resolve()
        require(source_path.is_file(), f'{where}: audio file not found: {source_path}')
        fingerprint = string(source.get('sha256'), f'{where}.sha256')
        require(re.fullmatch('[0-9a-f]{64}', fingerprint) is not None, f'{where}: invalid SHA256')
        hasher = hashlib.sha256()
        with source_path.open('rb') as audio:
            for block in iter(lambda: audio.read(1024 * 1024), b''):
                hasher.update(block)
        require(hasher.hexdigest() == fingerprint, f'{where}: audio SHA256 mismatch')
        offset = number(source.get('offset'), f'{where}.offset')
        length = number(source.get('duration'), f'{where}.duration')
        require(length > 0 and offset >= 0 and abs(offset - duration) <= EPS, f'{where}: source offsets must be continuous')
        if 'sample_count' in source or 'sample_rate' in source:
            count, rate = source.get('sample_count'), source.get('sample_rate')
            require(type(count) is int and count > 0 and type(rate) is int and rate > 0,
                    f'{where}: sample_count and sample_rate must both be positive integers')
            require(abs(length - count / rate) <= EPS, f'{where}: duration differs from PCM sample count')
        duration += length
        source['path'] = str(source_path)  # Saved manifest stays resolvable in its new directory.
    if 'pcm' in run:
        pcm = Path(string(run['pcm'], 'pcm'))
        pcm = (path.parent / pcm).resolve() if not pcm.is_absolute() else pcm.resolve()
        require(pcm.is_file(), f'pcm: canonical PCM not found: {pcm}')
        try:
            with wave.open(str(pcm), 'rb') as audio:
                shape, frames = (audio.getnchannels(), audio.getsampwidth(), audio.getframerate()), audio.getnframes()
                present = 0  # The header's frame count is a claim; count the bytes actually on disk.
                while block := audio.readframes(1 << 20):
                    present += len(block)
        except (wave.Error, EOFError) as error:
            raise Invalid(f'pcm: unreadable WAV: {error}')
        require(shape == (1, 2, 16000), 'pcm: canonical PCM must be mono 16-bit 16 kHz WAV')
        require(present == frames * 2, f'pcm: header declares {frames} frames but data holds {present // 2}')
        require(abs(frames / 16000 - duration) <= EPS, 'pcm: sample count differs from source durations')
        run['pcm'] = str(pcm)
    speakers = run.get('speakers')
    require(isinstance(speakers, dict), 'speakers: object required (anonymous IDs map to {})')
    for key, speaker in speakers.items():
        string(key, 'speaker ID')
        require(isinstance(speaker, dict), f'speakers.{key}: object required')
        if speaker:
            string(speaker.get('name'), f'speakers.{key}.name')
            require(speaker.get('status') in ('confirmed', 'possible'), f'speakers.{key}.status: confirmed or possible required')
            string(speaker.get('evidence'), f'speakers.{key}.evidence')
    candidates = run.get('participant_candidates', [])
    require(isinstance(candidates, list), 'participant_candidates: array required')
    for candidate in candidates:
        string(candidate, 'participant candidate')
    turns = run.get('turns')
    require(isinstance(turns, list), 'turns: array required')
    previous = -1
    for i, turn in enumerate(turns):
        start, end = interval(turn, f'turns[{i}]', duration)
        require(start >= previous and end > start, f'turns[{i}]: nonempty turns must be sorted by start')
        require(isinstance(turn.get('speaker'), str) and turn['speaker'] in speakers, f'turns[{i}]: unknown speaker ID')
        previous = start
    chunks = run.get('chunks')
    require(isinstance(chunks, list) and chunks, 'chunks: nonempty array required')
    cursor, word_count = 0, 0
    for i, chunk in enumerate(chunks):
        where = f'chunks[{i}]'
        start, end = interval(chunk, where, duration)
        require(end > start and abs(start - cursor) <= EPS, f'{where}: chunks must continuously cover the full recording')
        require(chunk.get('complete') is True and chunk.get('truncated') is False,
                f'{where}: require complete=true and truncated=false from inference completion evidence')
        text = string(chunk.get('text'), f'{where}.text', nonempty=False)
        words = chunk.get('words')
        require(isinstance(words, list), f'{where}: missing word alignment')
        character_ends, characters = set(), 0
        for char in text:
            characters += len(normalized(char))
            character_ends.add(characters)
        previous_start, previous_end, word_text, aligned_characters = start, start, '', 0
        for j, word in enumerate(words):
            wstart, wend = interval(word, f'{where}.words[{j}]', duration)
            require(start - EPS <= wstart <= wend <= end + EPS
                    and wstart >= previous_start - EPS and wend >= previous_end - EPS,
                    f'{where}.words[{j}]: alignment outside chunk or out of order')
            token = string(word.get('text'), f'{where}.words[{j}].text')
            token_text = normalized(token)
            require(bool(token_text), f'{where}.words[{j}]: attach standalone punctuation to an adjacent word')
            aligned_characters += len(token_text)
            require(aligned_characters in character_ends,
                    f'{where}.words[{j}]: alignment splits or exceeds a source character; regroup or realign')
            word_text += token
            previous_start, previous_end = wstart, wend
        require(normalized(word_text) == normalized(text), f'{where}: missing alignment or word text differs from original')
        word_count += len(words)
        cursor = end
    require(abs(cursor - duration) <= EPS, 'chunks do not cover the recording end')
    report = {'ok': True, 'duration': duration, 'sources': len(sources), 'chunks': len(chunks), 'words': word_count,
              'source_sha256_verified': True, 'pcm_verified': 'pcm' in run,
              'diarization_complete': run.get('diarization_complete') is True,
              'text_match': 'ignore Unicode punctuation, whitespace and case only',
              'limits': 'Structural validation and recorded completion flags cannot prove ASR accuracy or unheard speech.'}
    return run, report


def classify(word, turns):
    """Conservative word attribution: never let a sentence majority erase an interjection."""
    # ponytail: O(words * turns); use an interval sweep if long-run postprocessing dominates.
    start, end = word['start'], word['end']
    intersections = [(max(start, t['start']), min(end, t['end']), t['speaker'])
                     for t in turns if t['start'] < end and t['end'] > start]
    ids = tuple(sorted({speaker for _, _, speaker in intersections}))
    if not ids:
        return 'unknown', ()
    # Different voices active at the same instant, not merely a turn boundary.
    for i, (a, b, speaker) in enumerate(intersections):
        if any(other != speaker and min(b, d) > max(a, c) for c, d, other in intersections[i + 1:]):
            return 'overlap', ids
    if len(ids) > 1:
        return 'uncertain', ids
    # Union intervals so duplicated turns never inflate confidence.
    covered, cursor = 0.0, start
    for a, b, _ in sorted(intersections):
        covered += max(0.0, b - max(cursor, a))
        cursor = max(cursor, b)
    return ('speaker' if covered > 0 and covered >= (end - start) * 0.5 else 'uncertain'), ids


def pieces(chunk):
    """Map aligned tokens back onto exact original characters, retaining punctuation."""
    text = chunk['text']
    positions = [i for i, c in enumerate(text) if not c.isspace() and not unicodedata.category(c).startswith('P')
                 for _ in c.casefold()]
    consumed, cursor = 0, 0
    for word in chunk['words']:
        consumed += len(normalized(word['text']))
        boundary = positions[consumed] if consumed < len(positions) else len(text)
        if consumed < len(positions):
            for i in range(positions[consumed - 1] + 1, boundary):
                if unicodedata.category(text[i]) in ('Ps', 'Pi'):
                    boundary = i  # Opening quotes/brackets belong with the following word.
                    break
        yield word, text[cursor:boundary]
        cursor = boundary


def rows(run):
    for chunk in run['chunks']:
        current = None
        if not chunk['words'] and chunk['text']:
            yield {'start': chunk['start'], 'end': chunk['end'], 'kind': 'unknown', 'speakers': (), 'text': chunk['text']}
        for word, text in pieces(chunk):
            kind, speakers = classify(word, run['turns'])
            if current is not None and (current['kind'], current['speakers']) == (kind, speakers):
                current['text'] += text
                current['end'] = max(current['end'], word['end'])
            else:
                if current is not None:
                    yield current
                current = {'start': word['start'], 'end': word['end'], 'kind': kind, 'speakers': speakers, 'text': text}
        if current is not None:
            yield current


def millis(seconds):
    return round(seconds * 1000)


def stamp(seconds):
    ms = millis(seconds)
    hours, ms = divmod(ms, 3600000)
    minutes, ms = divmod(ms, 60000)
    seconds, ms = divmod(ms, 1000)
    return f'{hours:02}:{minutes:02}:{seconds:02}.{ms:03}'


def label(row, speakers):
    names = []
    for key in row['speakers']:
        info = speakers[key]
        names.append(f'{key}｜可能是 {info["name"]}' if info.get('status') == 'possible'
                     else f'{key}｜{info["name"]}' if info else key)
    joined = '、'.join(names)
    return {'unknown': '未辨識說話者', 'uncertain': f'說話者不確定：{joined}',
            'overlap': f'重疊發言：{joined}', 'speaker': joined}[row['kind']]


def markdown(text):
    return re.sub(r'([\\`*_{}\[\]<>()#+.!|~-])', r'\\\1', text)


def clayprint_dir():
    """Sibling install first, then the current project's Claude/Codex skill folders."""
    for root in (Path(__file__).resolve().parents[2], Path.cwd() / '.claude/skills', Path.cwd() / '.agents/skills'):
        if (root / 'kerf-clayprint/scripts/theme-css.js').is_file():
            return root / 'kerf-clayprint'
    return None


def clayprint_css():
    """Active Clayprint design-layer CSS, or none when Clayprint/node is absent (page-local rules carry var() fallbacks)."""
    found, node = clayprint_dir(), shutil.which('node')
    if found and node:
        result = subprocess.run([node, str(found / 'scripts/theme-css.js')], capture_output=True, text=True,
                                encoding='utf-8', check=False)
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout
    return ''


def srt_stamp(seconds):
    return stamp(seconds).replace('.', ',')


def srt_groups(rendered, sources):
    """Group display cues only; preserve reader rows and all source timestamps.

    A collapsed row needs an adjacent positive cue in the same unambiguous source.
    Inclusive containment deliberately rejects zero-length rows on source seams.
    """
    if not rendered:
        return [], 'no rows'
    owners, positive = [], []
    for i, row in enumerate(rendered):
        start, end = row['start'], row['end']
        if not (math.isfinite(start) and math.isfinite(end) and 0 <= start <= end):
            return [], f'row {i + 1}: invalid display interval'
        matches = [j for j, source in enumerate(sources)
                   if source['offset'] <= start <= end <= source['offset'] + source['duration']]
        owners.append(matches[0] if len(matches) == 1 else None)
        positive.append(millis(end) > millis(start))
    groups, i = [], 0
    while i < len(rendered):
        if positive[i]:
            groups.append([i])
            i += 1
            continue
        first, owner = i, owners[i]
        if owner is None:
            return [], f'row {i + 1}: collapsed after millisecond rounding; source boundary or source ownership ambiguous'
        while i < len(rendered) and not positive[i] and owners[i] == owner:
            i += 1
        block = list(range(first, i))
        if groups and owners[groups[-1][-1]] == owner:
            groups[-1].extend(block)
        elif i < len(rendered) and positive[i] and owners[i] == owner:
            groups.append(block + [i])
            i += 1
        else:
            return [], f'row {first + 1}: collapsed after millisecond rounding; no adjacent positive cue in the same source'
    return groups, None


def reader_time(seconds):
    """Keep the legacy millisecond spelling only when it loses no precision."""
    short = f'{seconds:.3f}'
    return short if float(short) == seconds else str(seconds)


def reader_html(run, rendered, segments, reader_chunks):
    """Display-only grouping. One text span per row, bounded by ASR chunk/source."""
    if not rendered:
        return ''.join(segments)
    paragraphs = []
    for chunk_index, (chunk, first, last) in enumerate(reader_chunks):
        groups = []
        # A chunk can straddle a source seam. Never join rows across that seam;
        # ambiguous or spanning rows stand alone without guessing ownership.
        for i in range(first, last):
            row = rendered[i]
            owners = [j for j, source in enumerate(run['sources'])
                      if source['offset'] <= row['start'] <= row['end'] <= source['offset'] + source['duration']]
            owner = owners[0] if len(owners) == 1 else None
            if groups and owner is not None and groups[-1][0] == owner:
                groups[-1][1].append(i)
            else:
                groups.append((owner, [i]))
        for _, indices in groups:
            group = [rendered[i] for i in indices]
            start, end = min(row['start'] for row in group), max(row['end'] for row in group)
            brief = f'{stamp(start).split(".")[0]}–{stamp(end).split(".")[0]}'
            paragraphs.append(f'<div class="chunk" data-chunk="{chunk_index}">'
                              f'<p class="chunk-meta">{brief}</p>'
                              f'<p class="chunk-text">{"".join(segments[i] for i in indices)}</p></div>')
    return ''.join(paragraphs)


def page(title, rows_html, markdown_text, meta, playback, exports=None):
    """Portable reader; optional formal exports preserve the five-argument caller."""
    exports = exports if exports is not None else {'md': {'filename': 'transcript.md', 'type': 'text/markdown', 'text': markdown_text}}
    export_payload = json.dumps(exports, ensure_ascii=False).replace('<', '\\u003c')
    downloads = ''.join(f'<a data-download="{kind}" download="{item["filename"]}">下載{title}</a>'
                        for kind, title in [('txt', '純文字'), ('md', ' Markdown'), ('srt', ' SRT')]
                        if (item := exports.get(kind)))
    payload = json.dumps(markdown_text, ensure_ascii=False).replace('<', '\\u003c')
    subject = urllib.parse.quote(f'逐字稿：{title}')
    audio = '<audio controls preload="metadata" src="playback.wav"></audio>' if playback else ''
    meta_html = ''.join(f'<div><dt>{html.escape(k)}</dt><dd>{html.escape(v)}</dd></div>' for k, v in meta)
    return f"""<!doctype html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>逐字稿：{html.escape(title)}</title>
<style id="clayprint-theme">
{clayprint_css()}
</style>
<style>
/* Local layout only; colors and typography come from Clayprint when installed. */
html {{ color: var(--cp-ink, #141413); background: var(--cp-paper, #FAF9F5);
  font: var(--cp-body, 16px)/1.65 var(--cp-font, "Helvetica Neue", "Segoe UI", Arial, "PingFang TC", "Microsoft JhengHei", sans-serif); }}
body {{ margin: 0 auto; max-width: 940px; padding: 0 24px 72px; overflow-wrap: anywhere; }}
* {{ box-sizing: border-box; }}
.reader-header {{ padding: 24px 0 12px; }}
.reader-header h1 {{ margin: 0; font-size: clamp(24px, 4vw, 32px); line-height: 1.3; }}
.eyebrow {{ margin: 0 0 8px; color: var(--cp-accent, #C2553A); font-size: var(--cp-small, 14px); }}
.reader-tools {{ display: flex; flex-wrap: wrap; gap: 8px 24px; border-bottom: 1px solid var(--cp-line, #E8E6DC); }}
.reader-tools details {{ min-width: 0; max-width: 100%; }}
summary {{ cursor: pointer; min-height: 44px; padding: 10px 0; color: var(--cp-accent, #C2553A); }}
.reader-meta {{ margin: 0 0 12px; font-size: var(--cp-small, 14px); }}
.reader-meta dt {{ font-weight: 600; }}
.reader-meta dd {{ margin: 0 0 8px; overflow-wrap: anywhere; }}
#transcript {{ margin-top: 20px; }}
.player {{ position: sticky; top: 0; z-index: 1; display: flex; flex-wrap: wrap; gap: 12px; align-items: center;
  padding: 12px 0; background: var(--cp-paper, #fff); border-bottom: 1px solid var(--cp-line, #ddd); }}
.player audio {{ width: 100%; min-width: 0; }}
@media (max-width: 640px), (max-height: 500px) {{ .player {{ position: static; }} }}
@media (max-width: 640px) {{ body {{ padding-inline: 18px; }} }}
@media print {{ .reader-tools, .player {{ display: none; }} }}
.export-actions {{ display: flex; flex-wrap: wrap; gap: 8px; padding-bottom: 12px; }}
.export-actions button, .export-actions a {{ font: inherit; font-size: var(--cp-small, 14px); display: inline-flex; align-items: center; min-height: 44px; padding: 6px 12px; border: 1px solid var(--cp-accent, #C2553A);
  border-radius: 4px; background: none; color: var(--cp-accent, #C2553A); cursor: pointer; text-decoration: none; }}
button:focus-visible, a:focus-visible, summary:focus-visible, .seg:focus-visible {{ outline: 2px solid var(--cp-accent, #C2553A); outline-offset: 3px; }}
.chunk {{ margin: 24px 0; }}
.chunk-meta {{ margin: 0 0 8px; color: var(--cp-muted, #666); font-size: var(--cp-small, 14px); }}
.chunk-text {{ margin: 0; max-width: none; line-height: 1.9; overflow-wrap: anywhere; }}
.seg {{ display: inline; white-space: pre-wrap; }}
.seg[role="button"] {{ cursor: pointer; }}
.seg[role="button"]:hover, .seg:focus-visible, .seg.now {{ background: var(--cp-tint, #eee); }}
.sr-only {{ position: absolute; width: 1px; height: 1px; padding: 0; margin: -1px;
  overflow: hidden; clip: rect(0, 0, 0, 0); white-space: nowrap; border: 0; }}
</style>
</head>
<body>
<main>
<header class="reader-header">
<p class="eyebrow">Wordprint 逐字稿</p>
<h1>{html.escape(title)}</h1>
</header>
<div class="reader-tools">
<details><summary>匯出</summary><div class="export-actions">
<button type="button" id="copy-text">複製文字</button><button type="button" id="copy">複製 Markdown</button>
{downloads}<a href="mailto:?subject={subject}">寄信</a></div><p id="export-status" role="status"></p></details>
<details><summary>詳細資料</summary><dl class="reader-meta">{meta_html}</dl></details>
</div>
{f'<div class="player">{audio}<p id="playback-error" role="status" hidden>播放音檔無法載入。請保留 HTML 與 playback.wav 在同一資料夾，或回到對話重新提供播放資產。</p></div>' if playback else ''}
<section id="transcript" aria-label="逐字稿">
{rows_html}
</section>
</main>
<script type="application/json" id="markdown">{payload}</script>
<script type="application/json" id="exports">{export_payload}</script>
<script>
const audio = document.querySelector('audio');
const segs = [...document.querySelectorAll('[data-start]')];
const seek = (el) => {{ if (audio) {{ audio.currentTime = Number(el.dataset.start); audio.play().catch(() => {{}}); }} }};
segs.filter(() => audio).forEach((el) => {{
  el.addEventListener('click', () => seek(el));
  el.addEventListener('keydown', (e) => {{ if (e.key === 'Enter' || e.key === ' ') {{ e.preventDefault(); seek(el); }} }});
}});
if (audio) {{
  const showPlaybackError = () => {{ document.getElementById('playback-error').hidden = false; }};
  audio.addEventListener('error', showPlaybackError);
  if (audio.error) showPlaybackError();
}}
if (audio) audio.addEventListener('timeupdate', () => {{
  segs.forEach((el) => el.classList.toggle('now', audio.currentTime >= Number(el.dataset.start) && audio.currentTime < Number(el.dataset.end)));
}});
const exportData = JSON.parse(document.getElementById('exports').textContent);
const status = document.getElementById('export-status');
document.querySelectorAll('[data-download]').forEach((link) => {{
  const item = exportData[link.dataset.download];
  link.href = URL.createObjectURL(new Blob([item.text], {{type: item.type + ';charset=utf-8'}}));
}});
const copyExport = async (kind) => {{
  try {{ await navigator.clipboard.writeText(exportData[kind].text); status.textContent = '已複製'; }}
  catch {{ status.textContent = '瀏覽器不允許複製，請改用下載。'; }}
}};
document.getElementById('copy').addEventListener('click', () => copyExport('md'));
const copyText = document.getElementById('copy-text');
copyText.hidden = !exportData.txt;
copyText.addEventListener('click', () => copyExport('txt'));
</script>
</body>
</html>
"""


def render(run, report, output, original_script=False, opencc='opencc', title=None):
    output = Path(output).resolve()
    require(not output.exists(), f'output already exists; choose a new directory: {output}')
    require(output.parent.is_dir(), 'output parent directory does not exist')
    # Record chunk membership while generating rows; no time-based reconstruction.
    rendered, reader_chunks = [], []
    for chunk in run['chunks']:
        chunk_rows = list(rows(dict(run, chunks=[chunk])))
        reader_chunks.append((chunk, len(rendered), len(rendered) + len(chunk_rows)))
        rendered.extend(chunk_rows)
    texts = [row['text'] for row in rendered]
    if not original_script:
        binary = shutil.which(opencc)
        try:  # Windows has no OpenCC CLI package; the torch worker's environment ships the official Python binding.
            module = None if binary else __import__('opencc').OpenCC('s2twp.json')
        except ImportError:
            module = None
        require(binary or module, 'OpenCC not found; install it or explicitly pass --original-script')
        # Convert only transcript text; identity labels and source evidence remain untouched.
        if module:
            texts = [module.convert(text) for text in texts]
        elif texts:
            # Newlines isolate OpenCC's line context; choose framing absent from every row.
            index = 0
            token = f'__WORDPRINT_OPENCC_{index}__'
            while any(token in text for text in texts):
                index += 1
                token = f'__WORDPRINT_OPENCC_{index}__'
            separator = f'\n{token}\n'
            payload = separator + separator.join(texts) + separator
            # Split bytes before universal-newline decoding: a row ending in CR must
            # not consume the framing LF. Match the old text=True result per row.
            result = subprocess.run([binary, '-c', 's2twp.json'], input=payload.encode('utf-8'),
                                    stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)
            require(result.returncode == 0, f'OpenCC failed: {result.stderr.decode("utf-8").strip()}')
            parts = result.stdout.split(separator.encode('ascii'))
            require(len(parts) == len(texts) + 2 and parts[0] == parts[-1] == b'',
                    'OpenCC failed: invalid batch boundaries or row count')
            converted = [part.decode('utf-8').replace('\r\n', '\n').replace('\r', '\n')
                         for part in parts[1:-1]]
            require(all(not source or target for source, target in zip(texts, converted)),
                    'OpenCC failed: empty converted row')
            texts = converted
    raw = '\n'.join(chunk['text'] for chunk in run['chunks']) + '\n'
    transcript, md = [], ['# 含時間與說話者逐字稿', '']
    segments, cues = [], []
    if not rendered:
        md.extend([EMPTY_TRANSCRIPT_NOTICE, ''])
        segments.append(f'<p role="status">{EMPTY_TRANSCRIPT_NOTICE}</p>')
    groups, srt_reason = srt_groups(rendered, run['sources'])
    srt_ok = srt_reason is None
    for i, (row, text) in enumerate(zip(rendered, texts), 1):
        who = label(row, run['speakers'])
        header = f'[{stamp(row["start"])}–{stamp(row["end"])}] {who}'
        transcript.append(f'{header}\n{text}\n')
        md.extend([f'**{markdown(header)}**', '', markdown(text), ''])
        action = ' role="button" tabindex="0"' if run.get('pcm') else ''
        segments.append(f'<span class="seg"{action} data-start="{reader_time(row["start"])}" data-end="{reader_time(row["end"])}">'
                        f'<span class="who sr-only">{html.escape(header)}</span>'
                        f'<span class="text">{html.escape(text).replace(chr(13), "&#13;")}</span></span>')
    for i, group in enumerate(groups, 1):
        start = min(rendered[j]['start'] for j in group)
        end = max(rendered[j]['end'] for j in group)
        lines = '\n'.join(f'[{label(rendered[j], run["speakers"])}] {texts[j].strip()}' for j in group)
        cues.append(f'{i}\n{srt_stamp(start)} --> {srt_stamp(end)}\n{lines}\n')
    grouped_rows = sum(len(group) for group in groups if len(group) > 1)
    title = title or '＋'.join(Path(s['path']).stem for s in run['sources'])
    playback = run.get('pcm')
    saved = dict(run, pcm=str(output / 'playback.wav')) if playback else run
    meta = [('模型', run['model']['id']), ('長度', stamp(report['duration'])), ('來源', f'{len(run["sources"])} 個檔案')]
    files = {'raw.txt': raw, 'raw.md': '# 模型原始逐字稿\n\n' + markdown(raw),
             'transcript.txt': '\n'.join(transcript), 'transcript.md': '\n'.join(md),
             'run.json': json.dumps(saved, ensure_ascii=False, indent=2) + '\n'}
    if srt_ok:
        files['transcript.srt'] = '\n'.join(cues)
    exports = {kind: {'filename': name, 'type': mime, 'text': files[name]}
               for kind, name, mime in [('txt', 'transcript.txt', 'text/plain'),
                                        ('md', 'transcript.md', 'text/markdown'),
                                        ('srt', 'transcript.srt', 'application/x-subrip')]
               if name in files}
    files['transcript.html'] = page(title, reader_html(run, rendered, segments, reader_chunks),
                                    files['transcript.md'], meta, playback, exports)
    final_report = dict(report, reading_script='original' if original_script else 'OpenCC s2twp.json',
                        empty_transcript=not rendered,
                        **({'content_warning': EMPTY_TRANSCRIPT_NOTICE} if not rendered else {}),
                        srt_written=srt_ok, **({} if srt_ok else {'srt_reason': srt_reason}),
                        srt_grouped_rows=grouped_rows,
                        **({'srt_grouping_note': '字幕將相鄰同來源列合併為顯示區間，逐列保留文字與說話者標記；原始逐詞時間未修改，並非已修復逐詞對齊。'} if grouped_rows else {}),
                        playback_written=bool(playback),
                        unknown_rows=sum(r['kind'] == 'unknown' for r in rendered),
                        uncertain_rows=sum(r['kind'] == 'uncertain' for r in rendered),
                        overlap_rows=sum(r['kind'] == 'overlap' for r in rendered))
    temp = Path(tempfile.mkdtemp(prefix='.transcript-', dir=output.parent))
    try:
        files['validation.json'] = json.dumps(final_report, ensure_ascii=False, indent=2) + '\n'
        for name, content in files.items():
            (temp / name).write_text(content, encoding='utf-8')
        if playback:
            shutil.copyfile(playback, temp / 'playback.wav')
        # mkdir claims the destination without overwriting a concurrently created directory.
        output.mkdir()
        try:
            for item in temp.iterdir():
                item.replace(output / item.name)
        except Exception:
            shutil.rmtree(output)
            raise
    finally:
        shutil.rmtree(temp)
    return final_report


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest='command', required=True)
    for name in ('validate', 'render'):
        command = sub.add_parser(name)
        command.add_argument('run', type=Path, help='normalized schema_version=1 run.json; source hashes rechecked')
        if name == 'render':
            command.add_argument('--output', type=Path, required=True, help='new output directory (parent must exist)')
            command.add_argument('--original-script', action='store_true', help='explicitly skip Traditional Chinese conversion')
            command.add_argument('--opencc', default='opencc', help='OpenCC command/path; defaults to opencc on PATH, s2twp.json')
            command.add_argument('--title', help='reading-page title; defaults to source file names')
    args = parser.parse_args(argv)
    try:
        run, report = validate(args.run)
        if args.command == 'render':
            report = render(run, report, args.output, args.original_script, args.opencc, args.title)
        print(json.dumps(report))  # ASCII-safe CLI JSON also works with legacy Windows stdout encodings.
        return 0
    except (Invalid, OSError, ValueError) as error:
        print(json.dumps({'ok': False, 'error': str(error)}), file=sys.stderr)
        return 1


if __name__ == '__main__':
    sys.exit(main())


In [ ]:
%%writefile torch_worker.py
#!/usr/bin/env python3
"""PyTorch worker for machines without the MLX worker (Windows, Linux, Intel Mac) and for the Colab notebook.

Qwen3-ASR + ForcedAligner via qwen-asr, speakers via NVIDIA Nemotron-3-Diarization through NeMo (no Hugging Face
token). Writes one normalized-run.json (references/run-format.md). The Colab notebook embeds this file verbatim
(tests/test_notebook.py keeps them in sync). Runs in its own process so the qwen-asr pin on transformers never meets
another version already loaded in a notebook session.
"""
import argparse
import importlib.metadata
import importlib.util
import json
import subprocess
import signal
import shutil
import sys
import unicodedata
import wave
from pathlib import Path

_runtime_spec = importlib.util.spec_from_file_location("wordprint_runtime", Path(__file__).with_name("worker_runtime.py"))
runtime = importlib.util.module_from_spec(_runtime_spec)
_runtime_spec.loader.exec_module(runtime)

_transcript_spec = importlib.util.spec_from_file_location("wordprint_transcript", Path(__file__).with_name("transcript.py"))
transcript = importlib.util.module_from_spec(_transcript_spec)
_transcript_spec.loader.exec_module(transcript)


ALIGNER = "Qwen/Qwen3-ForcedAligner-0.6B"
DIARIZER = "nvidia/Nemotron-3-Diarization"  # OpenMDW-1.1, not gated; up to 8 speakers.


def decode(name):
    """原始音檔 → 單聲道 16 kHz s16 PCM 位元組。"""
    return subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", name, "-ac", "1", "-ar", "16000", "-f", "s16le", "-"],
                          check=True, capture_output=True).stdout


def gap(text):
    """對時器略過的空白、標點、符號原樣併入鄰詞（時間不變）；略過真的字則不併入、不借鄰詞時間，讓 validate 擋下。"""
    if any(not c.isspace() and unicodedata.category(c)[0] not in "PS" for c in text):  # transcript.py normalized，另略符號
        print(f"警告：對時漏掉「{text}」；這份結果驗證不會通過", flush=True)
        return ""
    return text


def transcribe(asr, names, model, max_new_tokens, platform, dtype, directory=None):
    """逐來源 ASR／對時，保留來源邊界；qwen-asr 自行切分來源內的長音訊。"""
    import numpy as np
    pcm, sources = b"", []
    for name in names:
        before = runtime.digest(name)
        data = decode(name)
        after = runtime.digest(name)
        if before != after:
            raise ValueError(f"來源音檔在解碼期間被更動：{name}")
        if not data or len(data) % 2:
            raise ValueError("PCM 解碼不完整或沒有音訊。")
        sources.append({"path": str(Path(name).resolve()), "sha256": before, "duration": len(data) / 32000,
                        "offset": len(pcm) / 32000, "sample_count": len(data) // 2, "sample_rate": 16000})
        pcm += data
    samples = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768
    if directory:
        runtime.write_pcm(directory / "canonical.wav", pcm)
        runtime.save(directory / "sources.json", sources)
    run = {"schema_version": 1, "sources": sources, "platform": platform,
           "model": {"id": model, "size": model.rsplit("-", 1)[1], "backend": "qwen-asr " + importlib.metadata.version("qwen-asr"),
                     "language": None, "dtype": str(dtype), "max_new_tokens": max_new_tokens},
           "alignment_model": {"id": ALIGNER},
           "chunks": [{"start": source["offset"], "end": source["offset"] + source["duration"],
                       "text": "", "words": [], "complete": False, "truncated": False, "generated_tokens": []}
                      for source in sources],
           "turns": [], "diarization_complete": False, "speakers": {}}
    generated = []  # 每來源獨立記錄生成長度；碰到上限仍標 truncated，不推測停止原因。
    original = asr.model.generate

    def generate(*args, **kwargs):
        output = original(*args, **kwargs)
        generated.append(output.sequences.shape[1] - kwargs["input_ids"].shape[1])
        return output

    asr.model.generate = generate
    try:
        sample_start = 0
        for index, (source, chunk) in enumerate(zip(sources, run["chunks"])):
            generated.clear()
            sample_end = sample_start + source["sample_count"]
            try:
                result = asr.transcribe((samples[sample_start:sample_end], 16000), return_time_stamps=True)[0]
                chunk["text"] = result.text
                if index == 0:
                    run["model"]["language"] = result.language
                if directory:
                    # Preserve provider text before alignment processing or the next source can fail.
                    runtime.raw_text(directory, [part["text"] for part in run["chunks"]])
                words, cursor = [], 0
                for item in result.time_stamps or []:
                    word = {"text": item.text, "start": source["offset"] + item.start_time,
                            "end": source["offset"] + item.end_time}
                    found = result.text.find(item.text, cursor)
                    if found >= 0:
                        word["text"] = gap(result.text[cursor:found]) + item.text
                        cursor = found + len(item.text)
                    # 不符時保留原樣，讓 validate 如實擋下；不借用另一來源的字或時間。
                    words.append(word)
                if words:
                    words[-1]["text"] += gap(result.text[cursor:])
                chunk.update(words=words, complete=True)
            finally:
                chunk.update(truncated=max(generated, default=0) >= max_new_tokens, generated_tokens=list(generated))
                if directory:
                    # Pending sources remain complete=False; this is never an ASR checkpoint by itself.
                    runtime.save(directory / "asr-run.json", run)
            sample_start = sample_end
    finally:
        asr.model.generate = original
    return run, samples, pcm


def diarize(run, samples, device):
    """Nemotron 一次處理整段（內部串流切塊），說話者依首次發言編成 S1、S2…；失敗時保留無說話者的結果並說明原因。"""
    import torch
    try:
        from nemo.collections.asr.models import SortformerEncLabelModel
        from nemo.collections.asr.modules import transformer_encoder_utils
        from torch.nn.attention.flex_attention import flex_attention
        # ponytail: NeMo compiles flex_attention on CUDA, and Triton needs more shared memory than a T4 has; eager is
        # slower but diarization is small next to ASR. Restore the compiled kernel if big GPUs ever need the speed.
        transformer_encoder_utils._flex_attention_compiled = flex_attention
        model = SortformerEncLabelModel.from_pretrained(DIARIZER, map_location=device).to(torch.float32).eval()
        modules = model.sortformer_modules  # 模型卡的離線設定
        modules.chunk_len, modules.chunk_right_context, modules.fifo_len, modules.spkcache_update_period = 340, 40, 40, 300
        model._check_streaming_parameters()
        segments = model.diarize(audio=[samples], batch_size=1, sample_rate=16000)[0]
    except Exception as error:
        print(f"這次不分說話者（{type(error).__name__}：{error}）。", flush=True)
        run.setdefault("warnings", []).append(f"diarization 未完成：{type(error).__name__}：{error}")
        return
    order, duration, turns = {}, len(samples) / 16000, []
    for segment in sorted(segments, key=lambda s: float(s.split()[0])):
        start, end, label = segment.split()
        start, end = float(start), min(float(end), duration)
        if end > start:
            turns.append({"start": start, "end": end, "speaker": order.setdefault(label, f"S{len(order) + 1}")})
    run.update(turns=turns, speakers={speaker: {} for speaker in order.values()}, diarization_complete=True,
               diarization_model={"id": DIARIZER, "backend": "nemo-toolkit " + importlib.metadata.version("nemo-toolkit")})


def interrupt(_signum, _frame):
    raise KeyboardInterrupt


def validation_error(path, require_speakers=False):
    """Use the delivery validator for both checkpoint admission and completion."""
    try:
        transcript.validate(path, require_speakers=require_speakers)
    except (ValueError, OSError) as error:
        return str(error)
    return None


def execute(args, directory):
    directory = Path(directory).resolve()
    state = {"stage": "asr", "status": "running", "warnings": [], "network": "model libraries may fetch missing models"}
    runtime.state(directory, state)
    import torch
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    if args.fallback and device == "cpu" and args.model == "1.7B":
        args.model = "0.6B"
        print("沒有分配到 GPU：改用 0.6B（CPU 上 1.7B 太慢）；要 1.7B 請換成 GPU 執行階段後重跑。", flush=True)
    model, dtype = f"Qwen/Qwen3-ASR-{args.model}", torch.float16 if device != "cpu" else torch.float32
    config = {"backend": "qwen-asr", "model": model, "aligner": ALIGNER, "diarizer": DIARIZER,
              "max_new_tokens": args.max_new_tokens, "dtype": str(dtype), "sample_rate": 16000}
    if len(args.audio) > 1:
        config["asr_partition"] = "source-v1"
    state["model"] = model
    runtime.state(directory, state)
    if args.resume_from:
        sources, pcm, checkpoint, old_state = runtime.resume(args.resume_from, directory, args.audio, config)
        if "asr" not in checkpoint["stages"]:
            raise ValueError("PyTorch checkpoint 尚未完成 ASR／對齊，無法只續跑說話者。")
        run = json.loads((directory / "asr-run.json").read_text(encoding="utf-8"))
        run["sources"] = sources
        import numpy as np
        samples = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768
        state["resumed_from"] = str(Path(args.resume_from).resolve())
        print("重用既有 ASR／對齊，重新驗證後續跑說話者分析。", flush=True)
    else:
        from qwen_asr import Qwen3ASRModel
        asr = Qwen3ASRModel.from_pretrained(model, dtype=dtype, device_map=device, max_new_tokens=args.max_new_tokens,
                                            forced_aligner=ALIGNER, forced_aligner_kwargs={"dtype": dtype, "device_map": device})
        print("轉錄中…", flush=True)
        run, samples, pcm = transcribe(asr, args.audio, model, args.max_new_tokens, f"{args.platform} {device}", dtype, directory)
        run["pcm"] = str(directory / "canonical.wav")
        runtime.save(directory / "asr-run.json", run)
        runtime.checkpoint(directory, config, run["sources"], "decode", ["canonical.wav", "sources.json"])
        if validation_error(directory / "asr-run.json") is None:
            runtime.checkpoint(directory, config, run["sources"], "asr", ["asr-run.json"])
        del asr
    runtime.raw_text(directory, [chunk["text"] for chunk in run["chunks"]])
    run["pcm"] = str(directory / "canonical.wav")
    if args.pcm and Path(args.pcm).resolve() != directory / "canonical.wav":
        shutil.copyfile(directory / "canonical.wav", args.pcm)
        run["pcm"] = str(Path(args.pcm).resolve())
    # Publish the ASR result before any diarization work or cancellation.
    runtime.save(directory / "normalized-run.json", run)
    if Path(args.out).resolve() != directory / "normalized-run.json":
        runtime.save(Path(args.out), run)
    error = validation_error(directory / "normalized-run.json")
    if error is not None:
        state["warnings"].append(f"ASR／對齊驗證未通過：{error}")
        state.update(stage="partial", status="partial", retry_stage="asr")
        runtime.state(directory, state)
        return
    state["stage"] = "diarization"
    runtime.state(directory, state)
    print("分說話者中…", flush=True)
    diarize(run, samples, device)
    runtime.save(directory / "normalized-run.json", run)
    if Path(args.out).resolve() != directory / "normalized-run.json":
        runtime.save(Path(args.out), run)
    state["warnings"].extend(run.get("warnings", []))
    error = validation_error(directory / "normalized-run.json", require_speakers=True)
    complete = error is None
    if error is not None:
        state["warnings"].append(f"完整逐字稿驗證未通過：{error}")
    state.update(stage="complete" if complete else "partial", status="complete" if complete else "partial",
                 retry_stage=None if complete else "diarization")
    runtime.state(directory, state)
    print(args.out, "：", sum(len(chunk["words"]) for chunk in run["chunks"]), "詞，說話者", len(run["speakers"]) or "未標記", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--audio", required=True, action="append", help="repeat to merge recordings in this order")
    parser.add_argument("--out", required=True, help="normalized-run.json to write")
    parser.add_argument("--model", default="0.6B", choices=["0.6B", "1.7B"])
    parser.add_argument("--max-new-tokens", type=int, default=2048)
    parser.add_argument("--platform", default=sys.platform)
    parser.add_argument("--pcm", help="also write the canonical 16 kHz WAV here (sets run.pcm)")
    parser.add_argument("--resume-from", help="previous run directory containing an ASR checkpoint")
    parser.add_argument("--cpu-falls-back-to-0.6b", dest="fallback", action="store_true")
    args = parser.parse_args()
    output = Path(args.out).resolve()
    # Historical CLI allows arbitrary filenames in a populated folder (including Colab).
    directory = output.parent if output.name == "normalized-run.json" else output.with_suffix(".run")
    try:
        runtime.check_location(args.resume_from, directory)
        runtime.check_location(args.resume_from, output.parent)
        if args.pcm:
            runtime.check_location(args.resume_from, Path(args.pcm).resolve().parent)
        if output.exists() or (args.pcm and Path(args.pcm).exists()):
            raise FileExistsError("輸出已存在；請使用新的輸出路徑，既有檔案未改動。")
        directory.mkdir(parents=True, exist_ok=True)
        if any(directory.iterdir()):
            raise FileExistsError("請使用新的或空的轉錄目錄；既有目錄未改動。")
        with (directory / "state.json").open("x", encoding="utf-8") as file:
            json.dump({"stage": "start", "status": "running", "warnings": [], "available_outputs": [], "retry_stage": None}, file)
    except (OSError, ValueError) as error:
        print(str(error), file=sys.stderr)
        return 1
    signal.signal(signal.SIGTERM, interrupt)
    try:
        execute(args, directory)
        return 0
    except (Exception, KeyboardInterrupt) as error:
        state = json.loads((directory / "state.json").read_text())
        runtime.finish_error(directory, state, error)
        print(str(error), file=sys.stderr)
        return 130 if isinstance(error, KeyboardInterrupt) else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import tempfile

for names in groups:
    job = Path(tempfile.mkdtemp(prefix="wordprint-", dir=".")).resolve()
    out = str(job / f"normalized-run-{Path(names[0]).stem}-{job.name}.json")
    command = [sys.executable, "torch_worker.py", "--model", MODEL_SIZE, "--max-new-tokens", str(MAX_NEW_TOKENS),
               "--platform", "Google Colab" if files else "Jupyter", "--cpu-falls-back-to-0.6b", "--out", out]
    get_ipython().system(shlex.join(command + [arg for name in names for arg in ("--audio", name)]))
    if not Path(out).exists():
        raise RuntimeError(f"{names} 轉錄失敗，原因在上方訊息")
    if files:
        files.download(out)